## Step 3: Feature matrix construction

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import joblib
from sklearn.preprocessing import StandardScaler

# ==========================================
# 0. 硬件环境检测与目录准备
# ==========================================
# 确保在涉及任何张量或大模型操作时启用 RTX 4060
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 当前使用的计算设备: {device}")
# 可选：如果之前有残留显存，清理一下
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 确保目录结构存在（严格遵守项目目录约定）
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

# ==========================================
# 1. 读取数据
# ==========================================
input_path = '../data/02_finbert_features_added.csv'
print(f"📂 正在加载数据: {input_path}")
df = pd.read_csv(input_path)

# ==========================================
# 2. 特征衍生 (Feature Engineering)
# ==========================================
# 必须在标准化之前计算复合得分，保留“方向 × 强度”的真实物理量纲
df['llm_composite_score'] = df['sentiment_class'] * (df['action_score'] / 10.0)
print("✅ LLM复合情感因子 (llm_composite_score) 计算完成。")

# ==========================================
# 3. 实验分组定义 (Feature Grouping)
# ==========================================
G1_cols = [
    'open', 'close', 'high', 'low', 'volume', 'blocks-size', 'avg-block-size',
    'n-transactions-total', 'n-transactions-per-block', 'hash-rate', 'difficulty',
    'miners-revenue', 'transaction-fees-usd', 'n-unique-addresses', 'n-transactions',
    'estimated-transaction-volume-usd', 'total-bitcoins', 'market-cap', 'fng_value', 'cbbi_value'
]
G2_cols = G1_cols + ['finbert_sentiment_score']
G3_cols = G1_cols + ['llm_composite_score']
G3_M_cols = G1_cols + ['sentiment_class', 'action_class', 'action_score']

# 汇总所有涉及的特征列（使用 set 去重，然后转回 list）
all_feature_cols = list(set(G1_cols + G2_cols + G3_cols + G3_M_cols))
# 目标变量与时间戳（这部分绝对不参与标准化）
meta_cols = ['timestamp', 'target_sigma_t_plus_1']

# 最终应该保留的列（27列 = 1时间 + 1目标 + 20基础 + 1FinBERT + 1复合 + 3原始LLM）
final_columns = meta_cols + all_feature_cols

# 数据清理：仅保留需要的27列，丢弃 avg_next_price、文本等无关列
df = df[final_columns].copy()
print(f"✅ 数据清理完成，当前 DataFrame 维度: {df.shape} (预期列数为 27)")

# ==========================================
# 4. 时序划分与严格标准化 (Split & Scaling)
# ==========================================
# 时序划分：Train(7) : Val(1) : Test(2) - 严禁 Shuffle
n_samples = len(df)
train_end = int(n_samples * 0.7)
val_end = int(n_samples * 0.8)

# 划分索引区间
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"📊 数据集划分 (7:1:2): Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# 初始化 StandardScaler
scaler = StandardScaler()

# 【关键】仅使用 Train 阶段的数据拟合 (fit) scaler，防止未来信息泄露
scaler.fit(train_df[all_feature_cols])

# 对全量数据的特征部分进行 transform
df_scaled = df.copy()
df_scaled[all_feature_cols] = scaler.transform(df[all_feature_cols])
print("✅ 特征标准化完成 (timestamp与target未被缩放)。")

# ==========================================
# 5. 模型保存与数据导出
# ==========================================
# 保存归一化模型，供后续模型预测后对特征进行逆缩放或评估时使用
scaler_save_path = '../models/feature_scaler.pkl'
joblib.dump(scaler, scaler_save_path)
print(f"💾 Scaler 模型已保存至: {scaler_save_path}")

# 保存最终干净、标准化后的特征矩阵
output_csv_path = '../data/03_final_feature_matrix.csv'
# 按常用习惯排一下列顺序，把 timestamp 和 target 放在最前
ordered_cols = meta_cols + [col for col in final_columns if col not in meta_cols]
df_scaled = df_scaled[ordered_cols]

df_scaled.to_csv(output_csv_path, index=False)
print(f"💾 最终特征矩阵已成功导出至: {output_csv_path}")

✅ 当前使用的计算设备: cuda
📂 正在加载数据: ../data/02_finbert_features_added.csv
✅ LLM复合情感因子 (llm_composite_score) 计算完成。
✅ 数据清理完成，当前 DataFrame 维度: (2337, 27) (预期列数为 27)
📊 数据集划分 (7:1:2): Train=1635, Val=234, Test=468
✅ 特征标准化完成 (timestamp与target未被缩放)。
💾 Scaler 模型已保存至: ../models/feature_scaler.pkl
💾 最终特征矩阵已成功导出至: ../data/03_final_feature_matrix.csv


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 0. 全局作图环境与字体设置 (Windows 兼容)
# ==========================================
# 设置中文字体为黑体 (SimHei)，并解决负号显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
# 设置全局字体大小，适应论文排版
plt.rcParams['font.size'] = 12

# 确保结果保存目录存在
os.makedirs('../results', exist_ok=True)

# 读取包含原始量纲特征的数据集（使用02文件，以便描述性统计反映真实业务含义）
data_path = '../data/02_finbert_features_added.csv'
print(f"📂 正在加载数据以进行统计制图: {data_path}")
df = pd.read_csv(data_path)

# 计算 LLM 复合因子 (S_LLM)
df['llm_composite_score'] = df['sentiment_class'] * (df['action_score'] / 10.0)

# 如果缺失某些极端情况下的缺失值，进行前向填充
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True) # 兜底

# ==========================================
# 1. 核心特征分析 (图 3-6, 图 3-7, 图 3-8)
# ==========================================

# 【图 3-6】LLM 复合因子分布直方图
plt.figure(figsize=(8, 6))
sns.histplot(df['llm_composite_score'], bins=30, kde=True, color='#2c7bb6', edgecolor='black', alpha=0.7)
plt.xlabel('LLM复合情感因子 ($S_{LLM}$)', fontsize=14)
plt.ylabel('频数 (Frequency)', fontsize=14)
# 严禁 plt.title()
plt.tight_layout()
plt.savefig('../results/3_6_llm_composite_hist.png', dpi=300, format='png')
plt.close()
print("✅ 图 3-6 (LLM 复合因子分布直方图) 已保存。")


# 【图 3-7】FinBERT (S_FB) vs LLM 复合因子 (S_LLM) 分布对比图 (双KDE)
plt.figure(figsize=(8, 6))
sns.kdeplot(df['finbert_sentiment_score'], label='FinBERT ($S_{FB}$)', fill=True, color='#d7191c', alpha=0.4, linewidth=2)
sns.kdeplot(df['llm_composite_score'], label='LLM复合因子 ($S_{LLM}$)', fill=True, color='#2c7bb6', alpha=0.4, linewidth=2)
plt.xlabel('情感得分 (Sentiment Score)', fontsize=14)
plt.ylabel('概率密度 (Density)', fontsize=14)
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.savefig('../results/3_7_finbert_vs_llm_kde.png', dpi=300, format='png')
plt.close()
print("✅ 图 3-7 (分布对比KDE图) 已保存。")


# 【图 3-8】最终特征相关性热力图
# 为了图表美观与可读性，抽取核心基础特征与情感因子进行相关性展示
core_cols_for_heatmap = [
    'target_sigma_t_plus_1', 'finbert_sentiment_score', 'llm_composite_score', 
    'close', 'volume', 'transaction-fees-usd', 'hash-rate', 'fng_value'
]
# 重命名列以便在热力图中显示更友好的标签
heatmap_labels = [
    '$\sigma_{t+1}$', '$S_{FB}$', '$S_{LLM}$', 
    'Close Price', 'Volume', 'Tx Fees', 'Hash Rate', 'FNG Index'
]

corr_matrix = df[core_cols_for_heatmap].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='RdYlBu_r', 
            xticklabels=heatmap_labels, yticklabels=heatmap_labels, 
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/3_8_feature_correlation_heatmap.png', dpi=300, format='png')
plt.close()
print("✅ 图 3-8 (特征相关性热力图) 已保存。")


# ==========================================
# 2. 时序领先性分析 (图 3-9)
# ==========================================

# 【图 3-9】情绪因子对波动率的滞后相关性柱状图
max_lags = 7
lags = np.arange(1, max_lags + 1)
# 计算 S_t 与 \sigma_{t+k} 的相关性
# 注意：数据集中的 target_sigma_t_plus_1 本身已经是 t+1 的波动率。
# 所以如果我们要测 t 对应 t+k (k=1..7) 的真实波动率，我们需要将 target_sigma_t_plus_1 再 shift -(k-1)
corr_llm = [df['llm_composite_score'].corr(df['target_sigma_t_plus_1'].shift(-(k-1))) for k in lags]
corr_fb = [df['finbert_sentiment_score'].corr(df['target_sigma_t_plus_1'].shift(-(k-1))) for k in lags]

plt.figure(figsize=(10, 6))
bar_width = 0.35
x_indexes = np.arange(len(lags))

plt.bar(x_indexes - bar_width/2, corr_fb, width=bar_width, label='FinBERT ($S_{FB}$)', color='#d7191c', alpha=0.8)
plt.bar(x_indexes + bar_width/2, corr_llm, width=bar_width, label='LLM复合因子 ($S_{LLM}$)', color='#2c7bb6', alpha=0.8)

plt.xlabel('滞后天数 ($k$ days)', fontsize=14)
plt.ylabel('与对数极值波动率 ($\sigma_{t+k}$) 的相关系数', fontsize=14)
plt.xticks(x_indexes, [f'Lag {k}' for k in lags])
plt.axhline(0, color='black', linewidth=1, linestyle='--')
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.savefig('../results/3_9_lagged_correlation.png', dpi=300, format='png')
plt.close()
print("✅ 图 3-9 (滞后相关性柱状图) 已保存。")


# ==========================================
# 3. 统计学实证表格 (表 3-8, 表 3-9, 表 3-10)
# ==========================================

# 【表 3-8】特征分组清单表
groups_data = {
    "实验组": ["G1 (基础对照组)", "G2 (传统NLP情感组)", "G3 (LLM复合情感组)", "G3-M (LLM多维情感组)"],
    "包含特征": [
        "20项基础市场与链上特征",
        "G1特征 + FinBERT日度情感指数 ($S_{FB}$)",
        "G1特征 + LLM复合情感因子 ($S_{LLM}$)",
        "G1特征 + sentiment_class, action_class, action_score"
    ],
    "特征维度": [20, 21, 21, 23]
}
df_groups = pd.DataFrame(groups_data)
df_groups.to_csv('../results/3_8_feature_groups.csv', index=False, encoding='utf-8-sig')
print("✅ 表 3-8 (特征分组清单表) 已导出。")


# 【表 3-9】全变量描述性统计表
# 提取所有即将入模的特征
G1_cols = [
    'open', 'close', 'high', 'low', 'volume', 'blocks-size', 'avg-block-size',
    'n-transactions-total', 'n-transactions-per-block', 'hash-rate', 'difficulty',
    'miners-revenue', 'transaction-fees-usd', 'n-unique-addresses', 'n-transactions',
    'estimated-transaction-volume-usd', 'total-bitcoins', 'market-cap', 'fng_value', 'cbbi_value'
]
all_used_cols = G1_cols + ['finbert_sentiment_score', 'llm_composite_score', 'target_sigma_t_plus_1']
df_desc = df[all_used_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
df_desc.columns = ['均值 (Mean)', '标准差 (Std)', '最小值 (Min)', '25%分位数', '中位数 (Median)', '75%分位数', '最大值 (Max)']
df_desc.index.name = '特征变量 (Feature)'
df_desc.to_csv('../results/3_9_descriptive_stats.csv', encoding='utf-8-sig')
print("✅ 表 3-9 (全变量描述性统计表) 已导出。")


# 【表 3-10】Granger 因果检验结果表
# 检验目标：情绪因子的历史值是否能预测当前的波动率
# 由于 target_sigma_t_plus_1 是 t+1 期的波动率，为符合经典检验逻辑 X_t -> Y_t，
# 我们将 target_sigma_t_plus_1 往前平移一格作为当期波动率 current_sigma
df_granger = pd.DataFrame({
    'current_sigma': df['target_sigma_t_plus_1'].shift(1),
    'S_LLM': df['llm_composite_score'],
    'S_FB': df['finbert_sentiment_score']
}).dropna()

maxlag = 3
results_list = []

for factor in ['S_LLM', 'S_FB']:
    # grangercausalitytests 测试第二列是否引起第一列 (factor -> current_sigma)
    test_res = grangercausalitytests(df_granger[['current_sigma', factor]], maxlag=maxlag, verbose=False)
    for lag in range(1, maxlag + 1):
        # 提取 SSR F test 的 p-value
        p_val = test_res[lag][0]['ssr_ftest'][1]
        f_val = test_res[lag][0]['ssr_ftest'][0]
        results_list.append({
            '检验关系': f'{factor} -> $\sigma$',
            '滞后阶数 (Lag)': lag,
            'F-Statistic': round(f_val, 4),
            'P-Value': round(p_val, 4),
            '显著性结论': '显著 (H1)' if p_val < 0.05 else '不显著 (H0)'
        })

df_granger_res = pd.DataFrame(results_list)
df_granger_res.to_csv('../results/3_10_granger_causality.csv', index=False, encoding='utf-8-sig')
print("✅ 表 3-10 (Granger因果检验结果表) 已导出。")
print("\n🎉 所有章节 3.5 & 3.7 的图表与统计分析代码执行完毕！图片已可插入 Word 中。")

📂 正在加载数据以进行统计制图: ../data/02_finbert_features_added.csv
✅ 图 3-6 (LLM 复合因子分布直方图) 已保存。
✅ 图 3-7 (分布对比KDE图) 已保存。
✅ 图 3-8 (特征相关性热力图) 已保存。
✅ 图 3-9 (滞后相关性柱状图) 已保存。
✅ 表 3-8 (特征分组清单表) 已导出。
✅ 表 3-9 (全变量描述性统计表) 已导出。
✅ 表 3-10 (Granger因果检验结果表) 已导出。

🎉 所有章节 3.5 & 3.7 的图表与统计分析代码执行完毕！图片已可插入 Word 中。


In [1]:
import pandas as pd

# ==========================================
# 加载标准化后的特征矩阵
# ==========================================
file_path = '../data/03_final_feature_matrix.csv'
df = pd.read_csv(file_path)

# ==========================================
# 执行 7:1:2 时序切分逻辑
# ==========================================
n_samples = len(df)
train_end = int(n_samples * 0.7)
val_end = int(n_samples * 0.8)

# 提取各集合
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

# ==========================================
# 统计各集合的时间范围与样本量
# ==========================================
summary_data = [
    {
        "数据集类型": "训练集 (Train Set)",
        "样本量": len(train_df),
        "占比": "70%",
        "开始日期": train_df['timestamp'].iloc[0],
        "结束日期": train_df['timestamp'].iloc[-1]
    },
    {
        "数据集类型": "验证集 (Validation Set)",
        "样本量": len(val_df),
        "占比": "10%",
        "开始日期": val_df['timestamp'].iloc[0],
        "结束日期": val_df['timestamp'].iloc[-1]
    },
    {
        "数据集类型": "测试集 (Test Set)",
        "样本量": len(test_df),
        "占比": "20%",
        "开始日期": test_df['timestamp'].iloc[0],
        "结束日期": test_df['timestamp'].iloc[-1]
    }
]

# 转换为 DataFrame 并在 Notebook 中美化显示
split_summary_table = pd.DataFrame(summary_data)

# 打印表格
print("📊 毕业论文表 3-11: 数据集划分及其时间分布统计")
display(split_summary_table)

# 验证样本总数是否一致
print(f"✅ 验证结果: 总样本数({n_samples}) = 各部分之和({len(train_df) + len(val_df) + len(test_df)})")

📊 毕业论文表 3-11: 数据集划分及其时间分布统计


,数据集类型,样本量,占比,开始日期,结束日期
0,训练集 (Train Set),1635,70%,2018-02-01,2022-07-27
1,验证集 (Validation Set),234,10%,2022-07-28,2023-03-18
2,测试集 (Test Set),468,20%,2023-03-19,2024-06-28


✅ 验证结果: 总样本数(2337) = 各部分之和(2337)
